# Trabajo Práctico N° 1: Evaluación y Análisis de Redes Neuronales CBOW
**Asignatura**: Aprendizaje Automático Avanzado (UNAHUR)

Este notebook se destina a la visualización y análisis de resultados utilizando los modelos entrenados.

## Importación de librerías y módulos

In [ ]:
# Carga de librerias y módulos
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from codigo.modelo_cbow_cupy import cargar_modelo
from collections import Counter
from codigo.tokenizador import tokenizar_corpus

In [ ]:
def buscar_palabras_similares(
    palabra_buscada: str,
    W: np.ndarray,
    vocabulario_palabras: np.ndarray,
    top_k: int = 10,
    tipo_similaridad: str = "producto_interno"
) -> list[tuple[str, float]]:
    """
    Busca las palabras más similares a una palabra objetivo dada basándose en las filas de la matriz W.
    
    :param palabra_buscada: Palabra objetivo ingresada en texto.
    :param W: Matriz de pesos de entrada (|V| x N).
    :param vocabulario_palabras: Arreglo de cadenas de texto con las |V| palabras del vocabulario.
    :param top_k: Cantidad de vecinos más cercanos a retornar.
    :param tipo_similaridad: 'producto_interno' o 'coseno'.
    :return: Lista de tuplas (palabra, puntaje).
    """
    palabra_normalizada = palabra_buscada.lower().strip()
    vocab_lista = list(vocabulario_palabras)
    
    if palabra_normalizada not in vocab_lista:
        print(f"Advertencia: La palabra '{palabra_buscada}' no se encuentra en el vocabulario.")
        palabra_normalizada = "<UNK>"

    indice_objetivo = vocab_lista.index(palabra_normalizada)

    if hasattr(W, "get"):
        matriz_W = W.get()
    else:
        matriz_W = np.asarray(W)

    vector_palabra = matriz_W[indice_objetivo, :]

    if tipo_similaridad == "producto_interno":
        puntajes = np.dot(matriz_W, vector_palabra)
    elif tipo_similaridad == "coseno":
        norma_vector = np.linalg.norm(vector_palabra)
        normas_matriz = np.linalg.norm(matriz_W, axis=1)
        producto_punto = np.dot(matriz_W, vector_palabra)
        puntajes = producto_punto / (normas_matriz * norma_vector + 1e-12)
    else:
        raise ValueError(f"Tipo de similaridad no soportado: '{tipo_similaridad}'")

    indices_ordenados = np.argsort(puntajes)[::-1]

    resultados = []
    for indice_candidato in indices_ordenados:
        if indice_candidato == indice_objetivo:
            continue
        resultados.append((vocabulario_palabras[indice_candidato], float(puntajes[indice_candidato])))
        if len(resultados) >= top_k:
            break

    return resultados

## Carga de un Modelo Entrenado (.npz)
Carga la estructura completa del modelo desde un archivo  y grafica la función de pérdida.

In [ ]:
# Cargar el modelo resguardado en formato .npz
ruta_modelo = "respaldos/modelo_cbow_w4_epoca_500.npz"
modelo = cargar_modelo(ruta_modelo)

## Extracción de Vocabulario y Análisis de Distribución sobre el Corpus
Análisis de la estructura del vocabulario $|V|$ recuperado del modelo cargado y cálculo de las frecuencias de aparición de cada palabra sobre la secuencia completa del corpus de entrenamiento ().

In [ ]:
# 1. Extraer el vocabulario (palabras únicas |V|) del modelo cargado
vocabulario = modelo["vocabulario_palabras"]
tamanio_vocab = len(vocabulario)
print(f"Tamaño del vocabulario de palabras únicas del modelo (|V|): {tamanio_vocab:,}")

# 2. Cargar y tokenizar la secuencia COMPLETA del corpus de texto
ruta_corpus = modelo.get("configuracion", {}).get("ruta_corpus", "datos/corpus.txt")
tokens_completos = tokenizar_corpus(ruta_corpus=ruta_corpus, incluir_puntuacion_y_numeros=True)
print(f"Total de tokens en la secuencia del corpus completo: {len(tokens_completos):,}")

# 3. Contar ocurrencias de cada palabra sobre la secuencia completa del corpus
frecuencias_corpus = Counter(tokens_completos)

# Mapear frecuencias para las palabras presentes en el vocabulario
frecuencias_palabras = [frecuencias_corpus[p] for p in vocabulario if p != "<UNK>"]
longitudes_palabras = [len(str(p)) for p in vocabulario if p != "<UNK>"]

# Mostrar las 10 palabras más frecuentes en el corpus completo
top_10 = frecuencias_corpus.most_common(10)
print()
print("Top 10 palabras más frecuentes en el corpus completo:")
for pos, (palabra, freq) in enumerate(top_10, 1):
    print(f"  {pos:2d}. '{palabra}': {freq:,} apariciones")

# 4. Visualización Gráfica mediante Histogramas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma A: Distribución de Frecuencia de Aparición (Escala Logarítmica - Ley de Zipf)
sns.histplot(frecuencias_palabras, bins=40, log_scale=True, color="#2b5c8f", kde=True, ax=axes[0])
axes[0].set_title("Distribución de Frecuencias de Aparición (Corpus Completo)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Frecuencia de Ocurrencia (Escala Logarítmica)", fontsize=11)
axes[0].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)

# Histograma B: Distribución por Longitud de Caracteres del Vocabulario
sns.histplot(longitudes_palabras, discrete=True, color="#d95f02", ax=axes[1])
axes[1].set_title("Distribución por Longitud de Caracteres en Vocabulario", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Longitud de la Palabra (caracteres)", fontsize=11)
axes[1].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)

plt.tight_layout()
plt.show()


## Gráfico de Pérdida por Época

In [ ]:
# Graficar curva de pérdida por época
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
epocas = range(1, len(modelo["historial_perdida"]) + 1)
plt.plot(epocas, modelo["historial_perdida"], marker="o", color="#2b5c8f", linewidth=2.5, label="Pérdida CBOW")
plt.title("Evolución de la Pérdida de Entrenamiento por Época", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio (Cross-Entropy)", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Búsqueda de Similaridad de Palabras
Evalúa palabras similares calculando el producto interno o la similaridad de coseno directamente sobre las filas de la matriz $ y el arreglo .

In [ ]:
# Probar la búsqueda de similaridad sobre el modelo cargado (Producto Interno y Similitud Coseno)
palabra_test = "sol"

similares_pi = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="producto_interno"
)

similares_cos = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="coseno"
)

print(f"=== Palabras más similares a '{palabra_test}' (Producto Interno) ===")
for pal, score in similares_pi:
    print(f" - {pal:15s}: {score:.4f}")

print(f"\n=== Palabras más similares a '{palabra_test}' (Similitud Coseno) ===")
for pal, score in similares_cos:
    print(f" - {pal:15s}: {score:.4f}")


In [ ]:
# Cargar el modelo resguardado en formato .npz
ruta_modelo = "respaldos/modelo_cbow_w5_epoca_500.npz"
modelo = cargar_modelo(ruta_modelo)

## Extracción de Vocabulario y Análisis de Distribución sobre el Corpus
Análisis de la estructura del vocabulario $|V|$ recuperado del modelo cargado y cálculo de las frecuencias de aparición de cada palabra sobre la secuencia completa del corpus de entrenamiento ().

In [ ]:
# 1. Extraer el vocabulario (palabras únicas |V|) del modelo cargado
vocabulario = modelo["vocabulario_palabras"]
tamanio_vocab = len(vocabulario)
print(f"Tamaño del vocabulario de palabras únicas del modelo (|V|): {tamanio_vocab:,}")

# 2. Cargar y tokenizar la secuencia COMPLETA del corpus de texto
ruta_corpus = modelo.get("configuracion", {}).get("ruta_corpus", "datos/corpus.txt")
tokens_completos = tokenizar_corpus(ruta_corpus=ruta_corpus, incluir_puntuacion_y_numeros=True)
print(f"Total de tokens en la secuencia del corpus completo: {len(tokens_completos):,}")

# 3. Contar ocurrencias de cada palabra sobre la secuencia completa del corpus
frecuencias_corpus = Counter(tokens_completos)

# Mapear frecuencias para las palabras presentes en el vocabulario
frecuencias_palabras = [frecuencias_corpus[p] for p in vocabulario if p != "<UNK>"]
longitudes_palabras = [len(str(p)) for p in vocabulario if p != "<UNK>"]

# Mostrar las 10 palabras más frecuentes en el corpus completo
top_10 = frecuencias_corpus.most_common(10)
print()
print("Top 10 palabras más frecuentes en el corpus completo:")
for pos, (palabra, freq) in enumerate(top_10, 1):
    print(f"  {pos:2d}. '{palabra}': {freq:,} apariciones")

# 4. Visualización Gráfica mediante Histogramas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma A: Distribución de Frecuencia de Aparición (Escala Logarítmica - Ley de Zipf)
sns.histplot(frecuencias_palabras, bins=40, log_scale=True, color="#2b5c8f", kde=True, ax=axes[0])
axes[0].set_title("Distribución de Frecuencias de Aparición (Corpus Completo)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Frecuencia de Ocurrencia (Escala Logarítmica)", fontsize=11)
axes[0].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)

# Histograma B: Distribución por Longitud de Caracteres del Vocabulario
sns.histplot(longitudes_palabras, discrete=True, color="#d95f02", ax=axes[1])
axes[1].set_title("Distribución por Longitud de Caracteres en Vocabulario", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Longitud de la Palabra (caracteres)", fontsize=11)
axes[1].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)

plt.tight_layout()
plt.show()


## Gráfico de Pérdida por Época

In [ ]:
# Graficar curva de pérdida por época
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
epocas = range(1, len(modelo["historial_perdida"]) + 1)
plt.plot(epocas, modelo["historial_perdida"], marker="o", color="#2b5c8f", linewidth=2.5, label="Pérdida CBOW")
plt.title("Evolución de la Pérdida de Entrenamiento por Época", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio (Cross-Entropy)", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Búsqueda de Similaridad de Palabras
Evalúa palabras similares calculando el producto interno o la similaridad de coseno directamente sobre las filas de la matriz $ y el arreglo .

In [ ]:
# Probar la búsqueda de similaridad sobre el modelo cargado (Producto Interno y Similitud Coseno)
palabra_test = "sol"

similares_pi = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="producto_interno"
)

similares_cos = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="coseno"
)

print(f"=== Palabras más similares a '{palabra_test}' (Producto Interno) ===")
for pal, score in similares_pi:
    print(f" - {pal:15s}: {score:.4f}")

print(f"\n=== Palabras más similares a '{palabra_test}' (Similitud Coseno) ===")
for pal, score in similares_cos:
    print(f" - {pal:15s}: {score:.4f}")


## Comparación de Curvas de Pérdida entre Dos Modelos o Checkpoints (.npz)
Carga dos modelos entrenados en formato  y grafica de forma superpuesta sus curvas de pérdida.

In [ ]:
# Cargar dos modelos resguardados para comparativa
ruta_modelo_1 = "respaldos/modelo_cbow_w4_epoca_500.npz"
ruta_modelo_2 = "respaldos/modelo_cbow_w5_epoca_500.npz"

modelo_1 = cargar_modelo(ruta_modelo_1)
modelo_2 = cargar_modelo(ruta_modelo_2)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(modelo_1["historial_perdida"]) + 1), modelo_1["historial_perdida"], label="Modelo 1", color="#2b5c8f", linewidth=2)
plt.plot(range(1, len(modelo_2["historial_perdida"]) + 1), modelo_2["historial_perdida"], label="Modelo 2", color="#d95f02", linestyle="--", linewidth=2)
plt.title("Comparativa de Curvas de Pérdida entre Dos Modelos CBOW", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()
